# NEH on NEH

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut_point2/epoch_22.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
T_MC = 20
THRESHOLD = 0.5

# =====================================================
# 2. HELPER METRICS (UNCHANGED)
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

# =====================================================
# 3. DATASET (UNCHANGED)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform:
            img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 4. MODEL DEFINITION (MC DROPOUT)
# =====================================================
def get_resnet34_binary(pretrained=False, num_classes=2, p_drop=0.3):
    model = models.resnet34(pretrained=pretrained)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=p_drop),
        nn.Linear(in_features, num_classes)
    )
    return model

def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

print(f"\n--- Loading MC Dropout Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = get_resnet34_binary(pretrained=False, num_classes=2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
enable_mc_dropout(model)   # 🔑 keep dropout active

# =====================================================
# 5. MC DROPOUT INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference (MC Dropout)"):
        imgs = imgs.to(DEVICE)

        probs_T = []
        for _ in range(T_MC):
            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]
            probs_T.append(probs)

        probs_T = torch.stack(probs_T, dim=0)   # [T, B]
        probs_mean = probs_T.mean(dim=0)

        probs_all.extend(probs_mean.cpu().numpy())
        labels_all.extend(labels.numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= THRESHOLD).astype(int)

# =====================================================
# 6. METRICS (UNCHANGED)
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

spec = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 7. DISPLAY (UNCHANGED FORMAT)
# =====================================================
print("\n" + "=" * 60)
print(f"MODEL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")

print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {spec:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")
print(f"{'ECE':<30} | {ece:.4f}")

print("=" * 60)



--- Loading MC Dropout Model: epoch_22.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference (MC Dropout): 100%|██████████| 21/21 [00:24<00:00,  1.15s/it]


MODEL EVALUATION — epoch_22.pth
Metric                         | Value
------------------------------------------------------------
Accuracy                       | 0.8854
AUC                            | 0.9516
F1                             | 0.8779
F2 (β=2)                       | 0.8852
Macro F1                       | 0.8850
Macro F2 (β=2)                 | 0.8844
Precision                      | 0.9078
Recall                         | 0.8498
Macro Precision                | 0.8874
Macro Recall                   | 0.8844
Specificity                    | 0.9189
NPV                            | 0.8669
Cohen Kappa                    | 0.7702
------------------------------------------------------------
Set Size (avg, τ=0.9)          | 1.107
Singleton (%) (τ=0.9)          | 89.32
ECE                            | 0.0868


In [2]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. SETUP & PATHS
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut_point2/epoch_22.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut_point2/venn_abers_fitted.pkl"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
T_MC = 20

# =====================================================
# 3. HELPER METRICS FUNCTIONS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. MODEL DEFINITION & DATASET
# =====================================================
def get_resnet34_mc(num_classes=2, p_drop=0.3):
    model = models.resnet34(pretrained=False)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=p_drop),
        nn.Linear(in_features, num_classes)
    )
    return model

def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform: img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL & VENN-ABERS
# =====================================================
print(f"\n--- Loading MC Dropout Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = get_resnet34_mc(num_classes=2).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
enable_mc_dropout(model) # Keep dropout active for MC sampling

print(f"--- Loading Venn-Abers: {os.path.basename(VA_PICKLE_PATH)} ---")
with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 6. MC DROPOUT INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference (MC Dropout)"):
        imgs = imgs.to(DEVICE)
        
        batch_probs_T = []
        for _ in range(T_MC):
            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]
            batch_probs_T.append(probs)
        
        # Mean across T samples
        probs_mean = torch.stack(batch_probs_T, dim=0).mean(dim=0)
        
        probs_all.extend(probs_mean.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 7. METRIC COMPARISON
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"MC DROPOUT EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)


--- Loading MC Dropout Model: epoch_22.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference (MC Dropout): 100%|██████████| 21/21 [00:24<00:00,  1.18s/it]



Applying Venn-Abers Calibration...

MC DROPOUT EVALUATION: epoch_22.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.8839          | 0.8684         
AUC                            | 0.9517          | 0.9475         
F1                             | 0.8760          | 0.8682         
F2 (β=2)                       | 0.8836          | 0.8682         
Macro F1                       | 0.8834          | 0.8684         
Macro F2 (β=2)                 | 0.8828          | 0.8687         
Precision                      | 0.9075          | 0.8434         
Recall                         | 0.8466          | 0.8946         
Macro Precision                | 0.8860          | 0.8691         
Macro Recall                   | 0.8828          | 0.8692         
Specificity                    | 0.9189          | 0.8438         
NPV                            | 0.8644        

# NEH on UCSD

In [3]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut_point2/epoch_22.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
T_MC = 20
THRESHOLD = 0.5

# =====================================================
# 2. HELPER METRICS (UNCHANGED)
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

# =====================================================
# 3. DATASET (UNCHANGED)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform:
            img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 4. MODEL DEFINITION (MC DROPOUT)
# =====================================================
def get_resnet34_binary(pretrained=False, num_classes=2, p_drop=0.3):
    model = models.resnet34(pretrained=pretrained)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=p_drop),
        nn.Linear(in_features, num_classes)
    )
    return model

def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

print(f"\n--- Loading MC Dropout Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = get_resnet34_binary(pretrained=False, num_classes=2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
enable_mc_dropout(model)   # 🔑 keep dropout active

# =====================================================
# 5. MC DROPOUT INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference (MC Dropout)"):
        imgs = imgs.to(DEVICE)

        probs_T = []
        for _ in range(T_MC):
            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]
            probs_T.append(probs)

        probs_T = torch.stack(probs_T, dim=0)   # [T, B]
        probs_mean = probs_T.mean(dim=0)

        probs_all.extend(probs_mean.cpu().numpy())
        labels_all.extend(labels.numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= THRESHOLD).astype(int)

# =====================================================
# 6. METRICS (UNCHANGED)
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

spec = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 7. DISPLAY (UNCHANGED FORMAT)
# =====================================================
print("\n" + "=" * 60)
print(f"MODEL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")

print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {spec:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")
print(f"{'ECE':<30} | {ece:.4f}")

print("=" * 60)



--- Loading MC Dropout Model: epoch_22.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Inference (MC Dropout):   0%|          | 6/3014 [00:11<1:32:46,  1.85s/it]


KeyboardInterrupt: 

In [4]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. SETUP & PATHS
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut_point2/epoch_22.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut_point2/venn_abers_fitted.pkl"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
T_MC = 20

# =====================================================
# 3. HELPER METRICS FUNCTIONS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. MODEL DEFINITION & DATASET
# =====================================================
def get_resnet34_mc(num_classes=2, p_drop=0.3):
    model = models.resnet34(pretrained=False)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=p_drop),
        nn.Linear(in_features, num_classes)
    )
    return model

def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform: img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL & VENN-ABERS
# =====================================================
print(f"\n--- Loading MC Dropout Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = get_resnet34_mc(num_classes=2).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
enable_mc_dropout(model) # Keep dropout active for MC sampling

print(f"--- Loading Venn-Abers: {os.path.basename(VA_PICKLE_PATH)} ---")
with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 6. MC DROPOUT INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference (MC Dropout)"):
        imgs = imgs.to(DEVICE)
        
        batch_probs_T = []
        for _ in range(T_MC):
            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]
            batch_probs_T.append(probs)
        
        # Mean across T samples
        probs_mean = torch.stack(batch_probs_T, dim=0).mean(dim=0)
        
        probs_all.extend(probs_mean.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 7. METRIC COMPARISON
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"MC DROPOUT EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)


--- Loading MC Dropout Model: epoch_22.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference (MC Dropout): 100%|██████████| 3014/3014 [1:00:01<00:00,  1.19s/it]



Applying Venn-Abers Calibration...

MC DROPOUT EVALUATION: epoch_22.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.7580          | 0.6460         
AUC                            | 0.9399          | 0.9265         
F1                             | 0.8079          | 0.7467         
F2 (β=2)                       | 0.7448          | 0.6070         
Macro F1                       | 0.7404          | 0.5796         
Macro F2 (β=2)                 | 0.7362          | 0.5916         
Precision                      | 0.6944          | 0.5994         
Recall                         | 0.9658          | 0.9901         
Macro Precision                | 0.8135          | 0.7795         
Macro Recall                   | 0.7461          | 0.6264         
Specificity                    | 0.5264          | 0.2627         
NPV                            | 0.9325        

# NEH on DHU

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut/epoch_25.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
T_MC = 20
THRESHOLD = 0.5

# =====================================================
# 2. HELPER METRICS (UNCHANGED)
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

# =====================================================
# 3. DATASET (UNCHANGED)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform:
            img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 4. MODEL DEFINITION (MC DROPOUT)
# =====================================================
def get_resnet34_binary(pretrained=False, num_classes=2, p_drop=0.3):
    model = models.resnet34(pretrained=pretrained)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=p_drop),
        nn.Linear(in_features, num_classes)
    )
    return model

def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

print(f"\n--- Loading MC Dropout Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = get_resnet34_binary(pretrained=False, num_classes=2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
enable_mc_dropout(model)   # 🔑 keep dropout active

# =====================================================
# 5. MC DROPOUT INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference (MC Dropout)"):
        imgs = imgs.to(DEVICE)

        probs_T = []
        for _ in range(T_MC):
            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]
            probs_T.append(probs)

        probs_T = torch.stack(probs_T, dim=0)   # [T, B]
        probs_mean = probs_T.mean(dim=0)

        probs_all.extend(probs_mean.cpu().numpy())
        labels_all.extend(labels.numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= THRESHOLD).astype(int)

# =====================================================
# 6. METRICS (UNCHANGED)
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")

from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

spec = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 7. DISPLAY (UNCHANGED FORMAT)
# =====================================================
print("\n" + "=" * 60)
print(f"MODEL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")

print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {spec:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")
print(f"{'ECE':<30} | {ece:.4f}")

print("=" * 60)


In [5]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. SETUP & PATHS
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut_point2/epoch_22.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut_point2/venn_abers_fitted.pkl"

CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
T_MC = 20

# =====================================================
# 3. HELPER METRICS FUNCTIONS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. MODEL DEFINITION & DATASET
# =====================================================
def get_resnet34_mc(num_classes=2, p_drop=0.3):
    model = models.resnet34(pretrained=False)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=p_drop),
        nn.Linear(in_features, num_classes)
    )
    return model

def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform: img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL & VENN-ABERS
# =====================================================
print(f"\n--- Loading MC Dropout Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = get_resnet34_mc(num_classes=2).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
enable_mc_dropout(model) # Keep dropout active for MC sampling

print(f"--- Loading Venn-Abers: {os.path.basename(VA_PICKLE_PATH)} ---")
with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 6. MC DROPOUT INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference (MC Dropout)"):
        imgs = imgs.to(DEVICE)
        
        batch_probs_T = []
        for _ in range(T_MC):
            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]
            batch_probs_T.append(probs)
        
        # Mean across T samples
        probs_mean = torch.stack(batch_probs_T, dim=0).mean(dim=0)
        
        probs_all.extend(probs_mean.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 7. METRIC COMPARISON
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"MC DROPOUT EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)


# Applying Venn-Abers Calibration... 2nd run

# ===========================================================================
# MC DROPOUT EVALUATION: epoch_25.pth
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# Accuracy                       | 0.9358          | 0.9072         
# AUC                            | 0.9719          | 0.9717         
# F1                             | 0.9332          | 0.9125         
# F2 (β=2)                       | 0.9350          | 0.9066         
# Macro F1                       | 0.9357          | 0.9068         
# Macro F2 (β=2)                 | 0.9356          | 0.9061         
# Precision                      | 0.9908          | 0.8767         
# Recall                         | 0.8820          | 0.9513         
# Macro Precision                | 0.9405          | 0.9107         
# Macro Recall                   | 0.9368          | 0.9064         
# Specificity                    | 0.9915          | 0.8614         
# NPV                            | 0.8902          | 0.9447         
# Cohen Kappa                    | 0.8718          | 0.8140         
# Avg Set Size                   | 1.0771          | 1.5264         
# Singleton %                    | 92.2862         | 47.3647        
# ECE                            | 0.0395          | 0.0864         
# ===========================================================================

# Applying Venn-Abers Calibration...

# ===========================================================================
# MC DROPOUT EVALUATION: epoch_25.pth
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# Accuracy                       | 0.9354          | 0.9061         
# AUC                            | 0.9718          | 0.9715         
# F1                             | 0.9329          | 0.9116         
# F2 (β=2)                       | 0.9347          | 0.9056         
# Macro F1                       | 0.9353          | 0.9057         
# Macro F2 (β=2)                 | 0.9353          | 0.9050         
# Precision                      | 0.9900          | 0.8751         
# Recall                         | 0.8820          | 0.9513         
# Macro Precision                | 0.9401          | 0.9098         
# Macro Recall                   | 0.9364          | 0.9053         
# Specificity                    | 0.9908          | 0.8593         
# NPV                            | 0.8902          | 0.9445         
# Cohen Kappa                    | 0.8711          | 0.8119         
# Avg Set Size                   | 1.0785          | 1.5236         
# Singleton %                    | 92.1466         | 47.6440        
# ECE                            | 0.0400          | 0.0851         
# ===========================================================================



--- Loading MC Dropout Model: epoch_22.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference (MC Dropout): 100%|██████████| 90/90 [01:45<00:00,  1.17s/it]



Applying Venn-Abers Calibration...

MC DROPOUT EVALUATION: epoch_22.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.8366          | 0.6841         
AUC                            | 0.9674          | 0.9548         
F1                             | 0.8560          | 0.7612         
F2 (β=2)                       | 0.8329          | 0.6559         
Macro F1                       | 0.8336          | 0.6474         
Macro F2 (β=2)                 | 0.8314          | 0.6518         
Precision                      | 0.7762          | 0.6186         
Recall                         | 0.9540          | 0.9890         
Macro Precision                | 0.8569          | 0.7943         
Macro Recall                   | 0.8345          | 0.6786         
Specificity                    | 0.7150          | 0.3682         
NPV                            | 0.9376        

# NEH on OCT C8

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score
)

# =====================================================
# 1. SETUP
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut/epoch_25.pth"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
T_MC = 20
THRESHOLD = 0.5

# =====================================================
# 2. HELPER METRICS (UNCHANGED)
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask):
            continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

# =====================================================
# 3. DATASET (UNCHANGED)
# =====================================================
class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform:
            img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = UCSDDataset(CSV_PATH, test_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 4. MODEL DEFINITION (MC DROPOUT)
# =====================================================
def get_resnet34_binary(pretrained=False, num_classes=2, p_drop=0.3):
    model = models.resnet34(pretrained=pretrained)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=p_drop),
        nn.Linear(in_features, num_classes)
    )
    return model

def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

print(f"\n--- Loading MC Dropout Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = get_resnet34_binary(pretrained=False, num_classes=2)
model = model.to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
enable_mc_dropout(model)   # 🔑 keep dropout active

# =====================================================
# 5. MC DROPOUT INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference (MC Dropout)"):
        imgs = imgs.to(DEVICE)

        probs_T = []
        for _ in range(T_MC):
            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]
            probs_T.append(probs)

        probs_T = torch.stack(probs_T, dim=0)   # [T, B]
        probs_mean = probs_T.mean(dim=0)

        probs_all.extend(probs_mean.cpu().numpy())
        labels_all.extend(labels.numpy())

y_prob = np.asarray(probs_all)
y_true = np.asarray(labels_all)
y_pred = (y_prob >= THRESHOLD).astype(int)

# =====================================================
# 6. METRICS (UNCHANGED)
# =====================================================
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

f1 = f1_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

macro_precision = precision_score(y_true, y_pred, average="macro")
macro_recall = recall_score(y_true, y_pred, average="macro")
from sklearn.metrics import fbeta_score
f2 = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="weighted",
                zero_division=0
            )

f2_macro = fbeta_score(
                y_true,
                y_pred,
                beta=2.0,
                average="macro",
                zero_division=0
            )

spec = specificity_score(y_true, y_pred)
npv = npv_score(y_true, y_pred)

kappa = cohen_kappa_score(y_true, y_pred)

conf_metrics = set_size_and_singleton(y_prob, tau=0.9)
ece = expected_calibration_error(y_true, y_prob)

# =====================================================
# 7. DISPLAY (UNCHANGED FORMAT)
# =====================================================
print("\n" + "=" * 60)
print(f"MODEL EVALUATION — {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 60)

print(f"{'Metric':<30} | {'Value'}")
print("-" * 60)

print(f"{'Accuracy':<30} | {acc:.4f}")
print(f"{'AUC':<30} | {auc:.4f}")

print(f"{'F1':<30} | {f1:.4f}")
print(f"{'F2 (β=2)':<30} | {f2:.4f}")
print(f"{'Macro F1':<30} | {macro_f1:.4f}")
print(f"{'Macro F2 (β=2)':<30} | {f2_macro:.4f}")

print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")

print(f"{'Macro Precision':<30} | {macro_precision:.4f}")
print(f"{'Macro Recall':<30} | {macro_recall:.4f}")

print(f"{'Specificity':<30} | {spec:.4f}")
print(f"{'NPV':<30} | {npv:.4f}")

print(f"{'Cohen Kappa':<30} | {kappa:.4f}")

print("-" * 60)
print(f"{'Set Size (avg, τ=0.9)':<30} | {conf_metrics['avg_set_size']:.3f}")
print(f"{'Singleton (%) (τ=0.9)':<30} | {conf_metrics['singleton_rate']:.2f}")
print(f"{'ECE':<30} | {ece:.4f}")

print("=" * 60)


In [7]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. SETUP & PATHS
# =====================================================
CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut_point2/epoch_22.pth"
VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_MCDropOut_point2/venn_abers_fitted.pkl"
# CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered_imbalanced.csv"
CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv"


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
T_MC = 20

# =====================================================
# 3. HELPER METRICS FUNCTIONS
# =====================================================
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_pred = (y_proba >= 0.5).astype(int)
    confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not np.any(mask): continue
        acc_b = np.mean(y_pred[mask] == y_true[mask])
        conf_b = np.mean(confidence[mask])
        ece += np.abs(acc_b - conf_b) * np.mean(mask)
    return float(ece)

def calculate_all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Macro F2 (β=2)": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro"),
        "Specificity": specificity_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "Avg Set Size": conf_stats["avg_set_size"],
        "Singleton %": conf_stats["singleton_rate"],
        "ECE": expected_calibration_error(y_true, y_prob)
    }

# =====================================================
# 4. MODEL DEFINITION & DATASET
# =====================================================
def get_resnet34_mc(num_classes=2, p_drop=0.3):
    model = models.resnet34(pretrained=False)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=p_drop),
        nn.Linear(in_features, num_classes)
    )
    return model

def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

class UCSDDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["new_file_path"]).convert("RGB")
        label = int(row["binary_label"])
        if self.transform: img = self.transform(img)
        return img, label

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

loader = DataLoader(UCSDDataset(CSV_PATH, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# =====================================================
# 5. LOAD MODEL & VENN-ABERS
# =====================================================
print(f"\n--- Loading MC Dropout Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = get_resnet34_mc(num_classes=2).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
enable_mc_dropout(model) # Keep dropout active for MC sampling

print(f"--- Loading Venn-Abers: {os.path.basename(VA_PICKLE_PATH)} ---")
with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

# =====================================================
# 6. MC DROPOUT INFERENCE
# =====================================================
probs_all, labels_all = [], []

with torch.no_grad():
    for imgs, labels in tqdm(loader, desc="Inference (MC Dropout)"):
        imgs = imgs.to(DEVICE)
        
        batch_probs_T = []
        for _ in range(T_MC):
            logits = model(imgs)
            probs = F.softmax(logits, dim=1)[:, 1]
            batch_probs_T.append(probs)
        
        # Mean across T samples
        probs_mean = torch.stack(batch_probs_T, dim=0).mean(dim=0)
        
        probs_all.extend(probs_mean.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# 7. METRIC COMPARISON
# =====================================================
metrics_raw = calculate_all_metrics(y_true, y_prob_raw)
metrics_va = calculate_all_metrics(y_true, y_prob_va)

print("\n" + "=" * 75)
print(f"MC DROPOUT EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
print("=" * 75)
print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
print("-" * 75)

for key in metrics_raw.keys():
    print(f"{key:<30} | {metrics_raw[key]:<15.4f} | {metrics_va[key]:<15.4f}")

print("=" * 75)


# Applying Venn-Abers Calibration...

# ===========================================================================
# MC DROPOUT EVALUATION: epoch_9.pth
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# Accuracy                       | 0.5701          | 0.5722         
# AUC                            | 0.6759          | 0.6733         
# F1                             | 0.6617          | 0.6942         
# F2 (β=2)                       | 0.5525          | 0.5297         
# Macro F1                       | 0.5360          | 0.4912         
# Macro F2 (β=2)                 | 0.5432          | 0.5154         
# Precision                      | 0.5627          | 0.5547         
# Recall                         | 0.8030          | 0.9273         
# Macro Precision                | 0.5773          | 0.6247         
# Macro Recall                   | 0.5585          | 0.5545         
# Specificity                    | 0.3140          | 0.1818         
# NPV                            | 0.5919          | 0.6947         
# Cohen Kappa                    | 0.1196          | 0.1128         
# Avg Set Size                   | 1.3688          | 1.3604         
# Singleton %                    | 63.1175         | 63.9553        
# ECE                            | 0.3137          | 0.3004         
# ===========================================================================

# Applying Venn-Abers Calibration...

# ===========================================================================
# MC DROPOUT EVALUATION: epoch_25.pth
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# Accuracy                       | 0.8331          | 0.8274         
# AUC                            | 0.9082          | 0.9071         
# F1                             | 0.8285          | 0.8432         
# F2 (β=2)                       | 0.8319          | 0.8264         
# Macro F1                       | 0.8330          | 0.8256         
# Macro F2 (β=2)                 | 0.8339          | 0.8243         
# Precision                      | 0.8970          | 0.8039         
# Recall                         | 0.7697          | 0.8867         
# Macro Precision                | 0.8390          | 0.8317         
# Macro Recall                   | 0.8363          | 0.8244         
# Specificity                    | 0.9029          | 0.7622         
# NPV                            | 0.7810          | 0.8595         
# Cohen Kappa                    | 0.6679          | 0.6522         
# Avg Set Size                   | 1.1363          | 1.6066         
# Singleton %                    | 86.3676         | 39.3437        
# ECE                            | 0.1227          | 0.0597         
# ===========================================================================

# Applying Venn-Abers Calibration...

# ===========================================================================
# MC DROPOUT EVALUATION: epoch_22.pth
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# Accuracy                       | 0.8007          | 0.7656         
# AUC                            | 0.8369          | 0.7976         
# F1                             | 0.8613          | 0.8509         
# F2 (β=2)                       | 0.7967          | 0.7420         
# Macro F1                       | 0.7537          | 0.6514         
# Macro F2 (β=2)                 | 0.7442          | 0.6371         
# Precision                      | 0.8209          | 0.7523         
# Recall                         | 0.9058          | 0.9794         
# Macro Precision                | 0.7799          | 0.8126         
# Macro Recall                   | 0.7400          | 0.6421         
# Specificity                    | 0.5742          | 0.3049         
# NPV                            | 0.7388          | 0.8730         
# Cohen Kappa                    | 0.5105          | 0.3444         
# Avg Set Size                   | 1.2948          | 1.8310         
# Singleton %                    | 70.5226         | 16.8990        
# ECE                            | 0.1128          | 0.0628         
# ===========================================================================


--- Loading MC Dropout Model: epoch_22.pth ---


/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


--- Loading Venn-Abers: venn_abers_fitted.pkl ---


Inference (MC Dropout): 100%|██████████| 180/180 [03:35<00:00,  1.19s/it]



Applying Venn-Abers Calibration...

MC DROPOUT EVALUATION: epoch_22.pth
Metric                         | Before VA       | After VA       
---------------------------------------------------------------------------
Accuracy                       | 0.7427          | 0.6556         
AUC                            | 0.8268          | 0.7878         
F1                             | 0.7847          | 0.7476         
F2 (β=2)                       | 0.7358          | 0.6229         
Macro F1                       | 0.7325          | 0.6029         
Macro F2 (β=2)                 | 0.7302          | 0.6107         
Precision                      | 0.6984          | 0.6066         
Recall                         | 0.8953          | 0.9740         
Macro Precision                | 0.7658          | 0.7605         
Macro Recall                   | 0.7351          | 0.6398         
Specificity                    | 0.5749          | 0.3056         
NPV                            | 0.8332        

In [ ]:
# Applying Venn-Abers Calibration...

# ===========================================================================
# MC DROPOUT EVALUATION: epoch_25.pth
# ===========================================================================
# Metric                         | Before VA       | After VA       
# ---------------------------------------------------------------------------
# Accuracy                       | 0.8211          | 0.8540         
# AUC                            | 0.9160          | 0.9150         
# F1                             | 0.8567          | 0.8935         
# F2 (β=2)                       | 0.8204          | 0.8539         
# Macro F1                       | 0.8094          | 0.8307         
# Macro F2 (β=2)                 | 0.8259          | 0.8299         
# Precision                      | 0.9458          | 0.8904         
# Recall                         | 0.7830          | 0.8966         
# Macro Precision                | 0.8023          | 0.8321         
# Macro Recall                   | 0.8431          | 0.8294         
# Specificity                    | 0.9033          | 0.7622         
# NPV                            | 0.6589          | 0.7738         
# Cohen Kappa                    | 0.6243          | 0.6615         
# Avg Set Size                   | 1.1379          | 1.5661         
# Singleton %                    | 86.2137         | 43.3914        
# ECE                            | 0.1340          | 0.0416         
# ===========================================================================